# GRASP with path relinking for the ANN

This notebook demonstrates the implementation in `event_detection_pipeline/grasp.py`. The search minimizes the number of trainable ANN parameters while requiring validation **balanced accuracy** to be at least the score of the current `(128, 64)` ANN (within an optional tolerance).

Balanced accuracy is used instead of ordinary accuracy because contamination events are rare. A classifier that always predicts “no event” can have high ordinary accuracy but useless recall.

There is no fixed upper bound on hidden neurons during greedy construction. A finite `max_evaluations` budget remains necessary so an unreachable or noisy target cannot make the experiment run forever.

In [1]:
from pathlib import Path
import sys

# Make the project importable whether Jupyter starts in the repository root or src/notebooks.
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parents[1]
sys.path.insert(0, str(project_root))

from src.event_detection_pipeline.grasp import (
    GraspConfig,
    grasp_architecture_search,
    parameter_count,
)

## 1. Fast, transparent example

Training many ANNs is slow, so this first evaluator is a deterministic stand-in for validation balanced accuracy. It lets us inspect construction, local search, the elite pool, and path relinking immediately.

In [2]:
evaluated = []

def transparent_evaluator(hidden_sizes):
    # Saturating score with a small reward for distributing neurons across both layers.
    h1, h2 = hidden_sizes
    score = min(1.0, (h1 + 1.5 * h2) / 80.0)
    score += min(h1, h2) / 10_000.0
    evaluated.append((hidden_sizes, score))
    return score

config = GraspConfig(
    iterations=8,
    max_evaluations=60,   # runtime limit, not a neuron limit
    elite_size=6,
    relink_solutions=2,
    min_hidden_size=4,
    alpha=0.3,            # greediness/randomness balance for the RCL
    random_seed=42,
)

result = grasp_architecture_search(
    transparent_evaluator,
    input_dim=20,
    baseline_hidden_sizes=(128, 64),
    config=config,
)
result

GraspResult(hidden_sizes=(64, 64), metric=1.0064, baseline_metric=1.0064, parameter_count=5569, evaluations=27, elite=(ArchitectureScore(hidden_sizes=(64, 64), metric=1.0064, parameter_count=5569, feasible=True), ArchitectureScore(hidden_sizes=(128, 64), metric=1.0064, parameter_count=11009, feasible=True)))

In [3]:
print(f"Baseline metric:       {result.baseline_metric:.4f}")
print(f"Selected metric:       {result.metric:.4f}")
print(f"Selected architecture: {result.hidden_sizes}")
print(f"Selected parameters:   {result.parameter_count:,}")
print(f"Baseline parameters:   {parameter_count(20, (128, 64)):,}")
print(f"Unique evaluations:    {result.evaluations}")

print("\nElite pool:")
for item in result.elite:
    print(item)

Baseline metric:       1.0064
Selected metric:       1.0064
Selected architecture: (64, 64)
Selected parameters:   5,569
Baseline parameters:   11,009
Unique evaluations:    27

Elite pool:
ArchitectureScore(hidden_sizes=(64, 64), metric=1.0064, parameter_count=5569, feasible=True)
ArchitectureScore(hidden_sizes=(128, 64), metric=1.0064, parameter_count=11009, feasible=True)


## 2. Correspondence with Algorithm 6

- `construct()` starts at a small ANN and doubles one selected layer. Candidate moves form a restricted candidate list (RCL), from which one is chosen randomly.
- Feasibility means `metric >= baseline_metric - metric_tolerance`; construction therefore stops growing when it matches the ANN without GRASP.
- `local_search()` reduces layer widths while preserving feasibility.
- `path_relink()` moves one layer at a time from the new solution toward a randomly selected elite solution.
- `update_elite()` retains the smallest feasible architectures, breaking ties by better metric.
- The returned solution is the feasible elite member with the fewest trainable parameters.

The baseline itself is inserted into the elite pool. Thus the search can never return an architecture that fails the chosen target merely because its evaluation budget expires.

## 3. Real ANN evaluator on synthetic imbalanced event data

This cell uses the repository's `train_event_classifier`. Each architecture is trained with a deterministic seed, its alarm threshold is calibrated on validation data, and balanced accuracy is returned. Replace these arrays with the scaled classifier features assembled in `build_detector` to run the same search on the arsenic scenarios.

In [4]:
import numpy as np
import torch

from src.event_detection_pipeline.model import (
    predict_event_probability,
    select_device,
    train_event_classifier,
)
from src.event_detection_pipeline.pipeline import _calibrate_balanced_accuracy_threshold

rng = np.random.default_rng(42)
X = rng.normal(size=(1200, 12)).astype(np.float32)
signal = 1.8 * X[:, 0] - X[:, 1] + 0.7 * X[:, 2] * X[:, 3]
y = (signal > np.quantile(signal, 0.88)).astype(np.float32)
order = rng.permutation(len(X))
train_index, validation_index = order[:900], order[900:]
X_train, y_train = X[train_index], y[train_index]
X_validation, y_validation = X[validation_index], y[validation_index]
device = select_device()

trained_models = {}

def ann_evaluator(hidden_sizes):
    # Stable per-architecture seed makes cached comparisons reproducible.
    seed = 42 + sum((i + 1) * width for i, width in enumerate(hidden_sizes))
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = train_event_classifier(
        X_train, y_train, X_validation, y_validation, device,
        hidden_sizes=hidden_sizes,
        epochs=30,
        patience=5,
        batch_size=128,
    )
    probability = predict_event_probability(model, X_validation, device)
    threshold = _calibrate_balanced_accuracy_threshold(
        y_validation.astype(bool), probability
    )
    alarms = probability >= threshold
    flags = y_validation.astype(bool)
    sensitivity = alarms[flags].mean()
    specificity = (~alarms[~flags]).mean()
    trained_models[tuple(hidden_sizes)] = model
    return float(0.5 * (sensitivity + specificity))

In [5]:
# This cell trains several networks and can take a few minutes.
ann_result = grasp_architecture_search(
    ann_evaluator,
    input_dim=X_train.shape[1],
    baseline_hidden_sizes=(128, 64),
    config=GraspConfig(
        iterations=4,
        max_evaluations=20,
        elite_size=4,
        relink_solutions=1,
        min_hidden_size=4,
        # A small tolerance is often sensible because ANN training is noisy.
        metric_tolerance=0.005,
        random_seed=42,
    ),
)
ann_result

GraspResult(hidden_sizes=(32, 63), metric=0.9124513618677043, baseline_metric=0.9105510813501041, parameter_count=2559, evaluations=20, elite=(ArchitectureScore(hidden_sizes=(32, 63), metric=0.9124513618677043, parameter_count=2559, feasible=True), ArchitectureScore(hidden_sizes=(128, 64), metric=0.9105510813501041, parameter_count=9985, feasible=True)))

## 4. Production search on the arsenic detector

`build_detector` now performs this integration directly. It constructs the train/validation arrays once, caches every trained candidate, searches only on validation balanced accuracy, and installs the selected cached classifier in the final detector. The four held-out test scenarios are not passed to the search.

The saved `results/ANN/detector_32` configuration is used as the experiment baseline. Its `(32,)` architecture belongs to the chlorine-regression ANN; its `(128, 64)` event-classifier architecture is the baseline that GRASP resizes.

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.event_detection import (
    GraspConfig,
    build_detector,
    collect_default_data_paths,
    split_train_test,
)
from src.event_detection_pipeline.evaluation import (
    plot_chlorine_prediction_quality,
    plot_precision_recall_curve,
    plot_roc_curve_event_detection,
)
from src.event_detection_pipeline.model_io import save_detector

baseline_directory = project_root / "results" / "ANN" / "detector_32"
with (baseline_directory / "metadata.pkl").open("rb") as handle:
    baseline_config = pickle.load(handle)

print("Saved ANN configuration used as the GRASP baseline:")
for key in (
    "history", "epochs", "batch_size", "learning_rate", "hidden_sizes",
    "dropout", "weight_decay", "early_stopping_patience",
    "classifier_hidden_sizes", "classifier_dropout",
):
    print(f"  {key}: {baseline_config.get(key)}")

In [ ]:
data_paths = collect_default_data_paths(project_root / "src" / "data")
train_paths, test_paths = split_train_test(data_paths)

# Search expense is controlled by model evaluations, never by a neuron ceiling.
production_search_config = GraspConfig(
    iterations=8,
    max_evaluations=40,
    elite_size=6,
    relink_solutions=2,
    min_hidden_size=4,
    alpha=0.3,
    metric_tolerance=0.005,
    random_seed=42,
)

# This is the expensive cell: it trains the baseline and GRASP candidates.
optimal_detector = build_detector(
    train_paths,
    history=baseline_config["history"],
    epochs=baseline_config["epochs"],
    batch_size=baseline_config["batch_size"],
    learning_rate=baseline_config["learning_rate"],
    hidden_sizes=tuple(baseline_config["hidden_sizes"]),
    dropout=baseline_config["dropout"],
    weight_decay=baseline_config["weight_decay"],
    early_stopping_patience=baseline_config["early_stopping_patience"],
    max_validation_false_alarm_rate=baseline_config["max_validation_false_alarm_rate"],
    classifier_hidden_sizes=tuple(
        baseline_config.get("classifier_hidden_sizes", (128, 64))
    ),
    classifier_dropout=baseline_config.get("classifier_dropout", 0.1),
    grasp_config=production_search_config,
    random_seed=42,
)

save_directory = project_root / "results" / "ANN" / "grasp_optimal"
save_detector(optimal_detector, directory=str(save_directory))

In [ ]:
report = optimal_detector.grasp_report
print("GRASP optimal-solution report")
print("=" * 40)
for label in ("baseline", "selected"):
    values = report[label]
    print(f"\n{label.title()} model")
    for key, value in values.items():
        print(f"  {key}: {value}")
print(f"\nUnique architectures evaluated: {report['evaluations']}")
print(f"Training random seed: {report['random_seed']}")
print("Search budget and controls:")
for key, value in report["search_budget"].items():
    print(f"  {key}: {value}")
print("Classifier training parameters:")
for key, value in report["classifier_training"].items():
    print(f"  {key}: {value}")

print("\nFixed training parameters inherited from detector_32:")
training_parameters = {
    "regression_hidden_sizes": optimal_detector.hidden_sizes,
    "history": optimal_detector.history,
    "epochs": optimal_detector.epochs,
    "batch_size": optimal_detector.batch_size,
    "learning_rate": optimal_detector.learning_rate,
    "dropout": optimal_detector.dropout,
    "weight_decay": optimal_detector.weight_decay,
    "early_stopping_patience": optimal_detector.early_stopping_patience,
    "classifier_hidden_sizes": optimal_detector.classifier_hidden_sizes,
    "classifier_dropout": optimal_detector.classifier_dropout,
}
for key, value in training_parameters.items():
    print(f"  {key}: {value}")

## 5. Final held-out evaluation and the ANN notebook graphs

Only now, after GRASP has selected its architecture, is the optimal detector evaluated on `test_paths`. Each scenario remains separate so its timeline and event prevalence are not mixed with another scenario.

In [ ]:
def plot_metrics(result, title_prefix="", chlorine_indices=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    matrix = confusion_matrix(
        result.event_flags.astype(bool),
        result.alarms.astype(bool),
        labels=[False, True],
        normalize="all",
    )
    display_matrix = ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=["Normal", "Event"]
    )
    display_matrix.plot(ax=axes[0, 0], colorbar=False, values_format=".2f")
    axes[0, 0].set_title(f"{title_prefix}Confusion Matrix")
    plot_roc_curve_event_detection(
        result, title=f"{title_prefix}Event Detection ROC", ax=axes[0, 1]
    )
    plot_chlorine_prediction_quality(
        result,
        chlorine_indices=chlorine_indices,
        title=f"{title_prefix}Chlorine Prediction Quality",
        ax=axes[1, 0],
    )
    plot_precision_recall_curve(
        result, title=f"{title_prefix}Precision-Recall Curve", ax=axes[1, 1]
    )
    plt.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
test_results = [optimal_detector.detect(path) for path in test_paths]
summary = pd.DataFrame(
    [result.summary() for result in test_results],
    index=[path.name for path in test_paths],
)
display(summary)
display(pd.DataFrame(
    [summary.mean(numeric_only=True), summary.std(numeric_only=True)],
    index=["mean", "std"],
))

for index, (path, result) in enumerate(zip(test_paths, test_results), start=1):
    plot_metrics(
        result,
        title_prefix=(
            f"GRASP {optimal_detector.classifier_hidden_sizes} | "
            f"{path.name} | "
        ),
        chlorine_indices=list(range(result.actual.shape[1])),
    )